In [2]:
# Get descriptive statistics.

import zipfile
from PIL import Image
import io
import numpy as np

zip_path = "DeepDetect.zip"

res_real = []
res_fake = []

count = 0

with zipfile.ZipFile(zip_path, 'r') as z:
    files = [f for f in z.namelist()
             if f.lower().endswith((".jpg", ".jpeg", ".png"))
             and f.startswith("ddata/")]

    print(f"Found {len(files)} image files inside ZIP")

    for fname in files:
        count += 1

        if count % 10000 == 0:
            print(f"Processed {count} images...")

        # Determine class from folder path
        lower = fname.lower()
        if "/real/" in lower:
            label = "real"
        elif "/fake/" in lower:
            label = "fake"
        else:
            continue  # skip anything unexpected

        # Load image from ZIP
        try:
            with z.open(fname) as f:
                img = Image.open(io.BytesIO(f.read()))
                w, h = img.size
        except:
            print("Warning: could not open", fname)
            continue

        # Store resolution
        if label == "real":
            res_real.append((w, h))
        else:
            res_fake.append((w, h))

print("Done!\n")

# Convert to arrays
real_arr = np.array(res_real)
fake_arr = np.array(res_fake)

print("=== REAL ===")
print("Count:", len(real_arr))
print("Min:", real_arr.min(axis=0))
print("Max:", real_arr.max(axis=0))
print("Mean:", real_arr.mean(axis=0))

print("\n=== FAKE ===")
print("Count:", len(fake_arr))
print("Min:", fake_arr.min(axis=0))
print("Max:", fake_arr.max(axis=0))
print("Mean:", fake_arr.mean(axis=0))

Found 112185 image files inside ZIP
Processed 10000 images...
Processed 20000 images...
Processed 30000 images...
Processed 40000 images...
Processed 50000 images...
Processed 60000 images...
Processed 70000 images...
Processed 80000 images...
Processed 90000 images...
Processed 100000 images...
Processed 110000 images...
Done!

=== REAL ===
Count: 60192
Min: [102 150]
Max: [4616 3596]
Mean: [263.56082204 264.02636563]

=== FAKE ===
Count: 51991
Min: [134 105]
Max: [1152 1408]
Mean: [258.95066454 258.8088323 ]


In [3]:
# Count png images in each folder

import zipfile
from collections import defaultdict

zip_path = "DeepDetect.zip"

png_counts = defaultdict(int)

with zipfile.ZipFile(zip_path, 'r') as z:
    for fname in z.namelist():
        if not fname.lower().endswith(".png"):
            continue

        # fname looks like: ddata/train/real/123.png
        parts = fname.split("/")

        if len(parts) >= 4 and parts[0] == "ddata":
            subset = parts[1]   # train or test
            cls = parts[2]      # real or fake
            key = f"{subset}/{cls}"
            png_counts[key] += 1

# Print results
for key, count in png_counts.items():
    print(f"{key}: {count} PNG files")

# Total PNGs
print("\nTotal PNGs:", sum(png_counts.values()))

test/fake: 206 PNG files
test/real: 232 PNG files
train/fake: 873 PNG files

Total PNGs: 1311


In [ ]:
# Unify file types and image resolutions.

import zipfile
from PIL import Image
import io
import os

zip_path = "DeepDetect.zip"
output_dir = "dataset_2_cleaned"
target_size = (256, 256)

os.makedirs(output_dir, exist_ok=True)

count = 0

with zipfile.ZipFile(zip_path, 'r') as z:
    # Only include image files inside ddata/
    files = [
        f for f in z.namelist()
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
        and f.startswith("ddata/")
    ]

    print(f"Found {len(files)} image files inside ZIP")

    for fname in files:
        count += 1
        if count % 1000 == 0:
            print(f"Processed {count} images...")

        lower = fname.lower()

        # Determine class from folder path
        if "/real/" in lower:
            cls = "real"
        elif "/fake/" in lower:
            cls = "fake"
        else:
            # Skip anything unexpected
            continue

        class_dir = os.path.join(output_dir, cls)
        os.makedirs(class_dir, exist_ok=True)

        # Load image from ZIP
        try:
            with z.open(fname) as f:
                img = Image.open(io.BytesIO(f.read())).convert("RGB")
        except:
            print("Warning: could not open", fname)
            continue

        # Resize
        img = img.resize(target_size, Image.LANCZOS)

        # Base filename (strip extension)
        base = os.path.splitext(os.path.basename(fname))[0]

        # Add suffix to avoid collisions
        suffix = "_real" if cls == "real" else "_fake"
        out_name = base + suffix + ".jpg"
        out_path = os.path.join(class_dir, out_name)

        # Save as uniform JPG
        img.save(out_path, "JPEG", quality=95, subsampling=2)

print("Done! All images converted and resized.")

Found 112185 image files inside ZIP
Processed 1000 images...
Processed 2000 images...
Processed 3000 images...
Processed 4000 images...
Processed 5000 images...
Processed 6000 images...
Processed 7000 images...
Processed 8000 images...
Processed 9000 images...
Processed 10000 images...
Processed 11000 images...
Processed 12000 images...
Processed 13000 images...
Processed 14000 images...
Processed 15000 images...
Processed 16000 images...
Processed 17000 images...
Processed 18000 images...
Processed 19000 images...
Processed 20000 images...
Processed 21000 images...
Processed 22000 images...
Processed 23000 images...
Processed 24000 images...
Processed 25000 images...
Processed 26000 images...
Processed 27000 images...
Processed 28000 images...
Processed 29000 images...
Processed 30000 images...
Processed 31000 images...
Processed 32000 images...
Processed 33000 images...
Processed 34000 images...
Processed 35000 images...
Processed 36000 images...
Processed 37000 images...
Processed 3

In [5]:
# Validate updated dataset.

import os
from PIL import Image
import numpy as np

cleaned_path = "dataset_2_cleaned"

real_dir = os.path.join(cleaned_path, "real")
fake_dir = os.path.join(cleaned_path, "fake")

res_real = []
res_fake = []

# --- REAL IMAGES ---
real_files = [f for f in os.listdir(real_dir) if f.lower().endswith(".jpg")]

for i, fname in enumerate(real_files):
    if i % 10000 == 0:
        print(f"Processed {i} real images...")

    img_path = os.path.join(real_dir, fname)
    img = Image.open(img_path)
    w, h = img.size
    res_real.append((w, h))

# --- FAKE IMAGES ---
fake_files = [f for f in os.listdir(fake_dir) if f.lower().endswith(".jpg")]

for i, fname in enumerate(fake_files):
    if i % 10000 == 0:
        print(f"Processed {i} fake images...")

    img_path = os.path.join(fake_dir, fname)
    img = Image.open(img_path)
    w, h = img.size
    res_fake.append((w, h))

print("Done!")

# Convert to arrays for stats
real_arr = np.array(res_real)
fake_arr = np.array(res_fake)

print("=== REAL (cleaned JPG) ===")
print("Count:", len(real_arr))
print("Min:", real_arr.min(axis=0))
print("Max:", real_arr.max(axis=0))
print("Mean:", real_arr.mean(axis=0))

print("\n=== FAKE (cleaned JPG) ===")
print("Count:", len(fake_arr))
print("Min:", fake_arr.min(axis=0))
print("Max:", fake_arr.max(axis=0))
print("Mean:", fake_arr.mean(axis=0))

Processed 0 real images...
Processed 10000 real images...
Processed 20000 real images...
Processed 30000 real images...
Processed 40000 real images...
Processed 50000 real images...
Processed 60000 real images...
Processed 0 fake images...
Processed 10000 fake images...
Processed 20000 fake images...
Processed 30000 fake images...
Processed 40000 fake images...
Processed 50000 fake images...
Done!
=== REAL (cleaned JPG) ===
Count: 60192
Min: [256 256]
Max: [256 256]
Mean: [256. 256.]

=== FAKE (cleaned JPG) ===
Count: 51991
Min: [256 256]
Max: [256 256]
Mean: [256. 256.]


In [6]:
# Color comparison between real and fake images.

import os
import numpy as np
from PIL import Image

cleaned_path = "dataset_2_cleaned"
real_dir = os.path.join(cleaned_path, "real")
fake_dir = os.path.join(cleaned_path, "fake")

def compute_color_stats(folder):
    means = []
    stds = []

    files = [f for f in os.listdir(folder) if f.lower().endswith(".jpg")]

    for i, fname in enumerate(files):
        if i % 10000 == 0:
            print(f"Processed {i} images in {folder}...")

        img = Image.open(os.path.join(folder, fname)).convert("RGB")
        arr = np.array(img) / 255.0  # normalize to [0,1]

        means.append(arr.mean(axis=(0,1)))
        stds.append(arr.std(axis=(0,1)))

    means = np.array(means)
    stds = np.array(stds)

    return means.mean(axis=0), stds.mean(axis=0)

real_mean, real_std = compute_color_stats(real_dir)
fake_mean, fake_std = compute_color_stats(fake_dir)

print("\n=== REAL ===")
print("Mean RGB:", real_mean)
print("Std RGB:", real_std)

print("\n=== FAKE ===")
print("Mean RGB:", fake_mean)
print("Std RGB:", fake_std)

Processed 0 images in dataset_2_cleaned\real...
Processed 10000 images in dataset_2_cleaned\real...
Processed 20000 images in dataset_2_cleaned\real...
Processed 30000 images in dataset_2_cleaned\real...
Processed 40000 images in dataset_2_cleaned\real...
Processed 50000 images in dataset_2_cleaned\real...
Processed 60000 images in dataset_2_cleaned\real...
Processed 0 images in dataset_2_cleaned\fake...
Processed 10000 images in dataset_2_cleaned\fake...
Processed 20000 images in dataset_2_cleaned\fake...
Processed 30000 images in dataset_2_cleaned\fake...
Processed 40000 images in dataset_2_cleaned\fake...
Processed 50000 images in dataset_2_cleaned\fake...

=== REAL ===
Mean RGB: [0.51163522 0.42002649 0.37627459]
Std RGB: [0.25025372 0.22581356 0.22228649]

=== FAKE ===
Mean RGB: [0.51988021 0.42543724 0.38002325]
Std RGB: [0.2454115  0.22043857 0.2182703 ]
